# Orion Technical Assessment 

## EXTRACT — Clean


### Ingest & Clean Sales file

In [61]:
import pandas as pd , numpy as np 
import json , logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")

sales_path='data\\Sales.json'
forecast_path='data\\forecast.json'

In [62]:
def extract_data_from_json_files(path):

    logging.info(f"Reading {path[5:]}...")
    df = pd.read_json(path)

    logging.info(f"Extracted {len(df):,} records.")
    return df

In [63]:
sales_df=extract_data_from_json_files(sales_path)
sales_df.shape

2026-06-20 17:51:51,159 [INFO] Reading Sales.json...
2026-06-20 17:52:01,570 [INFO] Extracted 298,246 records.


(298246, 18)

In [64]:
sales_df.head(3)

,ProductKey,Product Name,Brand,Color,Subcategory,Category,CustomerKey,Customer Code,Name,Education,Occupation,Continent,City,State,CountryRegion,OrderDate,Quantity,Net Price
0,2492,Cigarette Lighter Adapter for Contoso Phones E...,Contoso,Cell phones Accessories,Cell phones Accessories,Cell phones,18813,CS551,None,None,None,North America,Spokane,Washington,United States,1/1/2008,1,23.7405
1,2492,Cigarette Lighter Adapter for Contoso Phones E...,Contoso,Cell phones Accessories,Cell phones Accessories,Cell phones,18813,CS551,None,None,None,North America,Spokane,Washington,United States,1/1/2008,1,23.7405
2,2492,Cigarette Lighter Adapter for Contoso Phones E...,Contoso,Cell phones Accessories,Cell phones Accessories,Cell phones,18813,CS551,None,None,None,North America,Spokane,Washington,United States,1/1/2008,1,23.7405


In [65]:
sales_df.dtypes

ProductKey         int64
Product Name      object
Brand             object
Color             object
Subcategory       object
Category          object
CustomerKey        int64
Customer Code     object
Name              object
Education         object
Occupation        object
Continent         object
City              object
State             object
CountryRegion     object
OrderDate         object
Quantity           int64
Net Price        float64
dtype: object

In [66]:
sales_df["OrderDate"] = pd.to_datetime(sales_df["OrderDate"], errors="coerce")

In [67]:
sales_df.isna().mean().sort_values(ascending=False)*100

Education        90.009254
Name             90.009254
Occupation       90.009254
ProductKey        0.000000
Product Name      0.000000
Brand             0.000000
Category          0.000000
Subcategory       0.000000
Customer Code     0.000000
Color             0.000000
CustomerKey       0.000000
Continent         0.000000
City              0.000000
State             0.000000
CountryRegion     0.000000
OrderDate         0.000000
Quantity          0.000000
Net Price         0.000000
dtype: float64

In [68]:
sales_df.drop(columns=['Education','Name','Occupation'],inplace=True)
logging.info(f"Dropping near-empty columns ..")

2026-06-20 17:52:02,721 [INFO] Dropping near-empty columns ..


In [69]:
print(f'The percentage of duplicated records is: {sales_df.duplicated().sum()} | {round(sales_df.duplicated().sum()/len(sales_df)*100,1)} %')

The percentage of duplicated records is: 218008 | 73.1 %


In [70]:
# Calculate revenue using all rows
revenue_all = (sales_df["Quantity"] * sales_df["Net Price"]).sum()

# Calculate revenue after removing duplicates (correct way)
sales_dedup = sales_df.drop_duplicates()
revenue_dedup = (sales_dedup["Quantity"] * sales_dedup["Net Price"]).sum()

print(f"Revenue keeping all rows: ${revenue_all:,.2f}")
print(f"Revenue after removing duplicates: ${revenue_dedup:,.2f}")

Revenue keeping all rows: $83,535,101.76
Revenue after removing duplicates: $42,644,968.84


In [71]:
# we should confirm with stakeholder before dropping duplicates !!!

# sales_df.drop_duplicates(inplace=True)

In [72]:
sales_df.head(3)

,ProductKey,Product Name,Brand,Color,Subcategory,Category,CustomerKey,Customer Code,Continent,City,State,CountryRegion,OrderDate,Quantity,Net Price
0,2492,Cigarette Lighter Adapter for Contoso Phones E...,Contoso,Cell phones Accessories,Cell phones Accessories,Cell phones,18813,CS551,North America,Spokane,Washington,United States,2008-01-01,1,23.7405
1,2492,Cigarette Lighter Adapter for Contoso Phones E...,Contoso,Cell phones Accessories,Cell phones Accessories,Cell phones,18813,CS551,North America,Spokane,Washington,United States,2008-01-01,1,23.7405
2,2492,Cigarette Lighter Adapter for Contoso Phones E...,Contoso,Cell phones Accessories,Cell phones Accessories,Cell phones,18813,CS551,North America,Spokane,Washington,United States,2008-01-01,1,23.7405


In [73]:
text_cols = ["Product Name", "Brand", "Color", "Subcategory", "Category",
                 "Customer Code", "Continent", "City", "State", "CountryRegion"]
for col in text_cols:
    sales_df[col] = sales_df[col].astype(str).str.strip()

logging.info(f"Trim whitespace on text columns ..")

2026-06-20 17:52:06,332 [INFO] Trim whitespace on text columns ..


In [74]:
# Check the percentage of valid date
print(f'Check the percentage of valid date : {100*(len(sales_df["OrderDate"].notna())) / len(sales_df)} %')

Check the percentage of valid date : 100.0 %


In [75]:
# Check for invalid Price or Quantity
print(f'No. of Invalid items(negative qty or price): {len(sales_df[(sales_df["Quantity"] < 0) | (sales_df["Net Price"] <= 0)])}')

No. of Invalid items(negative qty or price): 0


### Ingest & Clean forecast file

In [76]:
forecast_df=extract_data_from_json_files(forecast_path)
forecast_df.shape

2026-06-20 17:52:06,393 [INFO] Reading forecast.json...
2026-06-20 17:52:06,403 [INFO] Extracted 33 records.


(33, 4)

In [77]:
forecast_df.head()

,CountryRegion,Brand,Forecast,Year
0,China,A. Datum,201487,2009
1,China,Adventure Works,399490,2009
2,China,Contoso,1173308,2009
3,China,Fabrikam,790074,2009
4,China,Litware,437443,2009


In [78]:
forecast_df.dtypes

CountryRegion    object
Brand            object
Forecast          int64
Year              int64
dtype: object

In [79]:
# Trim whitespace on text columns 
forecast_df["Brand"] = forecast_df["Brand"].astype(str).str.strip()
forecast_df["CountryRegion"] = forecast_df["CountryRegion"].astype(str).str.strip()
logging.info(f"Trim whitespace on text columns ..")

2026-06-20 17:52:06,472 [INFO] Trim whitespace on text columns ..


In [80]:
# Drop Duplicates
forecast_df.drop_duplicates(inplace=True)

In [81]:
# Check for Null Values
forecast_df.isna().sum()

CountryRegion    0
Brand            0
Forecast         0
Year             0
dtype: int64

## Transform — split into star schema 

In [82]:
# Create Product Dimension

def build_products_dim(df_sales) :
    cols = ["ProductKey", "Product Name", "Brand", "Color", "Subcategory", "Category"]
    dim = df_sales[cols].drop_duplicates(subset="ProductKey").reset_index(drop=True)
    logging.info(f"Products dimension: {len(dim):,} unique products.")
    return dim

In [83]:
 # Create Customer Dimension
 
def build_customers_dim(df_sales ) :
    cols = ["CustomerKey", "Customer Code", "Continent", "City", "State", "CountryRegion"]
    dim = df_sales[cols].drop_duplicates(subset="CustomerKey").reset_index(drop=True)
    logging.info(f"Customers dimension: {len(dim):,} unique customers.")
    return dim

In [84]:
# Create Fact_Sales

def build_sales_fact(df_sales) :
    fact_sales = df_sales[["ProductKey", "CustomerKey", "OrderDate", "Quantity", "Net Price"]].copy()
    fact_sales["TotalAmount"] = fact_sales["Quantity"] * fact_sales["Net Price"]
    logging.info(f"Sales fact table: {len(fact_sales):,} rows.")
    return fact_sales

In [85]:
df_products = build_products_dim(sales_df)
df_customers = build_customers_dim(sales_df)
df_sales_fact = build_sales_fact(sales_df)
print(df_products.shape, df_customers.shape, df_sales_fact.shape)

2026-06-20 17:52:06,690 [INFO] Products dimension: 2,495 unique products.
2026-06-20 17:52:06,802 [INFO] Customers dimension: 8,868 unique customers.
2026-06-20 17:52:06,812 [INFO] Sales fact table: 298,246 rows.


(2495, 6) (8868, 6) (298246, 6)


In [86]:
df_products.head(3)

,ProductKey,Product Name,Brand,Color,Subcategory,Category
0,2492,Cigarette Lighter Adapter for Contoso Phones E...,Contoso,Cell phones Accessories,Cell phones Accessories,Cell phones
1,1763,MGS Flight Simulator 2000 M410,Tailspin Toys,Download Games,Download Games,Games and Toys
2,2124,Contoso Coffee Maker 12C M1000 Silver,Contoso,Coffee Machines,Coffee Machines,Home Appliances


In [87]:
df_customers.head(3)

,CustomerKey,Customer Code,Continent,City,State,CountryRegion
0,18813,CS551,North America,Spokane,Washington,United States
1,4751,15750,North America,Bellflower,California,United States
2,4760,15759,North America,Beaverton,Oregon,United States


In [88]:
df_sales_fact.head(3)

,ProductKey,CustomerKey,OrderDate,Quantity,Net Price,TotalAmount
0,2492,18813,2008-01-01,1,23.7405,23.7405
1,2492,18813,2008-01-01,1,23.7405,23.7405
2,2492,18813,2008-01-01,1,23.7405,23.7405


## Load — write to CSV 

In [89]:
df_products.to_csv("Outputs\\Products.csv",index=False)
df_customers.to_csv("Outputs\\Customers.csv",index=False)
df_sales_fact.to_csv("Outputs\\Sales.csv",index=False)
forecast_df.to_csv("Outputs\\Forecast.csv",index=False)